# AeroNetra — VisDrone Dataset Preparation

**Purpose:** Convert raw VisDrone-DET annotations to YOLO format for training.

**Kaggle Setup:**
1. Add dataset: `shisuiotsutsuki/visdrone2019-det` (or your VisDrone upload)
2. Accelerator: None (CPU is fine)
3. Internet: OFF (not needed)

**Outputs:** YOLO-format dataset saved to `/kaggle/working/visdrone_yolo/` — save as a Kaggle dataset to attach to training notebooks.

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================
from pathlib import Path
import os

# --- EDIT THIS to match your Kaggle dataset input path ---
VISDRONE_ROOT = Path("/kaggle/input/visdrone2019-det")

# Output directory (Kaggle working dir is saved as output)
OUTPUT_ROOT = Path("/kaggle/working/visdrone_yolo")

# Conversion mode: "merged" (all vehicles → class 0) or "separate" (8 vehicle classes)
MODE = "separate"

# Splits to convert
SPLITS = ["train", "val", "test"]

print(f"VisDrone root: {VISDRONE_ROOT}")
print(f"Output root:   {OUTPUT_ROOT}")
print(f"Mode:          {MODE}")
print(f"Root exists:   {VISDRONE_ROOT.exists()}")

In [ ]:
# ============================================================
# Cell 2: Discover the dataset directory structure
# ============================================================
# VisDrone datasets on Kaggle can have varying directory layouts.
# This cell auto-discovers the splits.

def discover_splits(root: Path) -> dict:
    """Discover VisDrone split directories (images + annotations)."""
    discovered = {}
    
    # Common patterns in VisDrone Kaggle uploads
    patterns = [
        # Pattern 1: VisDrone2019-DET-{split}/images, .../annotations
        ("VisDrone2019-DET-{split}", "images", "annotations"),
        # Pattern 2: {split}/images, {split}/annotations
        ("{split}", "images", "annotations"),
        # Pattern 3: Flat — images/{split}, annotations/{split}
        (None, "images/{split}", "annotations/{split}"),
    ]
    
    split_aliases = {
        "train": ["train", "VisDrone2019-DET-train"],
        "val": ["val", "VisDrone2019-DET-val"],
        "test": ["test", "test-dev", "VisDrone2019-DET-test-dev"],
    }
    
    for split, aliases in split_aliases.items():
        for alias in aliases:
            for parent_fmt, img_sub, ann_sub in patterns:
                if parent_fmt:
                    img_dir = root / parent_fmt.format(split=alias) / img_sub
                    ann_dir = root / parent_fmt.format(split=alias) / ann_sub
                else:
                    img_dir = root / img_sub.format(split=alias)
                    ann_dir = root / ann_sub.format(split=alias)
                
                if img_dir.exists() and ann_dir.exists():
                    discovered[split] = {"images": img_dir, "annotations": ann_dir}
                    break
            if split in discovered:
                break
    
    return discovered

# Discover
splits = discover_splits(VISDRONE_ROOT)

if not splits:
    # Fallback: list top-level contents for manual inspection
    print("Could not auto-discover splits. Directory contents:")
    for p in sorted(VISDRONE_ROOT.rglob("*")):
        if p.is_dir():
            print(f"  [DIR]  {p.relative_to(VISDRONE_ROOT)}")
    raise FileNotFoundError("Update VISDRONE_ROOT or split patterns above.")

for split, paths in splits.items():
    n_img = len(list(paths['images'].glob('*.jpg')))
    n_ann = len(list(paths['annotations'].glob('*.txt')))
    print(f"{split:6s}: {n_img:5d} images, {n_ann:5d} annotations")
    print(f"         images:      {paths['images']}")
    print(f"         annotations: {paths['annotations']}")

In [ ]:
# ============================================================
# Cell 3: VisDrone → YOLO converter (self-contained)
# ============================================================
# Inlined from src/aeronetra/datasets/visdrone.py

import cv2
import shutil
from typing import Dict, Optional, Tuple, List

# VisDrone class mapping
VISDRONE_VEHICLE_CLASSES = {
    3: "bicycle", 4: "car", 5: "van", 6: "truck",
    7: "tricycle", 8: "awning-tricycle", 9: "bus", 10: "motor",
}

_SEPARATE_MAP = {4: 0, 5: 1, 6: 2, 7: 3, 8: 4, 9: 5, 10: 6, 3: 7}

SEPARATE_CLASS_NAMES = {
    0: "car", 1: "van", 2: "truck", 3: "tricycle",
    4: "awning-tricycle", 5: "bus", 6: "motor", 7: "bicycle",
}
MERGED_CLASS_NAMES = {0: "vehicle"}


def parse_visdrone_row(row_str: str) -> Optional[Dict[str, int]]:
    """Parses a single VisDrone annotation line."""
    parts = row_str.strip().split(",")
    if len(parts) < 8:
        return None
    try:
        return {
            "left": int(parts[0]), "top": int(parts[1]),
            "width": int(parts[2]), "height": int(parts[3]),
            "score": int(parts[4]), "category": int(parts[5]),
            "truncation": int(parts[6]), "occlusion": int(parts[7]),
        }
    except ValueError:
        return None


def map_category(category: int, mode: str = "separate") -> Optional[int]:
    """Maps VisDrone category to YOLO class ID. Returns None for non-vehicle."""
    if category not in VISDRONE_VEHICLE_CLASSES:
        return None
    if mode == "merged":
        return 0
    if mode == "separate":
        return _SEPARATE_MAP.get(category)
    return None


def convert_box_to_yolo(
    box: Dict[str, int], img_w: int, img_h: int
) -> Optional[Tuple[float, float, float, float]]:
    """Converts VisDrone box to normalized YOLO (x_center, y_center, w, h)."""
    left = max(0, box["left"])
    top = max(0, box["top"])
    right = min(img_w, box["left"] + box["width"])
    bottom = min(img_h, box["top"] + box["height"])
    w, h = right - left, bottom - top
    if w <= 0 or h <= 0:
        return None
    xc = min(max((left + w / 2) / img_w, 0.0), 1.0)
    yc = min(max((top + h / 2) / img_h, 0.0), 1.0)
    return (xc, yc, min(w / img_w, 1.0), min(h / img_h, 1.0))


def convert_split(
    images_dir: Path, annotations_dir: Path, output_dir: Path, mode: str
) -> Dict[str, int]:
    """Converts one VisDrone split to YOLO format. Returns stats dict."""
    stats = {
        "total_images": 0, "converted": 0, "missing_labels": 0,
        "valid_annotations": 0, "ignored_annotations": 0,
        "malformed": 0, "skipped_zero_area": 0,
    }
    out_img = output_dir / "images"
    out_lbl = output_dir / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(images_dir.glob("*.jpg"))
    stats["total_images"] = len(image_paths)

    for img_path in image_paths:
        ann_path = annotations_dir / f"{img_path.stem}.txt"
        if not ann_path.exists():
            stats["missing_labels"] += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            stats["malformed"] += 1
            continue
        img_h, img_w = img.shape[:2]

        yolo_lines = []
        with open(ann_path, "r", encoding="utf-8") as f:
            for line in f:
                parsed = parse_visdrone_row(line)
                if not parsed:
                    stats["malformed"] += 1
                    continue
                cat_id = map_category(parsed["category"], mode)
                if cat_id is None:
                    stats["ignored_annotations"] += 1
                    continue
                yolo_box = convert_box_to_yolo(parsed, img_w, img_h)
                if not yolo_box:
                    stats["skipped_zero_area"] += 1
                    continue
                stats["valid_annotations"] += 1
                xc, yc, w, h = yolo_box
                yolo_lines.append(f"{cat_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

        # Write YOLO label
        with open(out_lbl / f"{img_path.stem}.txt", "w", encoding="utf-8") as f:
            f.writelines(yolo_lines)

        # Symlink image (saves disk space on Kaggle)
        dst = out_img / img_path.name
        if not dst.exists():
            try:
                os.symlink(img_path.resolve(), dst)
            except OSError:
                shutil.copy2(img_path, dst)

        stats["converted"] += 1

    return stats


print("Converter functions defined.")

In [ ]:
# ============================================================
# Cell 4: Run conversion for all splits
# ============================================================
import time

all_stats = {}

for split_name in SPLITS:
    if split_name not in splits:
        print(f"⚠️  Split '{split_name}' not found in dataset — skipping.")
        continue

    split_info = splits[split_name]
    out_dir = OUTPUT_ROOT / split_name

    print(f"\n{'='*60}")
    print(f"Converting: {split_name}")
    print(f"{'='*60}")

    t0 = time.time()
    stats = convert_split(
        images_dir=split_info["images"],
        annotations_dir=split_info["annotations"],
        output_dir=out_dir,
        mode=MODE,
    )
    elapsed = time.time() - t0

    all_stats[split_name] = stats
    print(f"  Images:      {stats['total_images']}")
    print(f"  Converted:   {stats['converted']}")
    print(f"  Valid annot: {stats['valid_annotations']}")
    print(f"  Ignored:     {stats['ignored_annotations']} (non-vehicle)")
    print(f"  Malformed:   {stats['malformed']}")
    print(f"  Zero-area:   {stats['skipped_zero_area']}")
    print(f"  Time:        {elapsed:.1f}s")

In [ ]:
# ============================================================
# Cell 5: Generate dataset.yaml for Ultralytics training
# ============================================================
import yaml

class_names = SEPARATE_CLASS_NAMES if MODE == "separate" else MERGED_CLASS_NAMES
nc = len(class_names)

dataset_yaml = {
    "path": str(OUTPUT_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images" if "test" in all_stats else "",
    "nc": nc,
    "names": class_names,
}

yaml_path = OUTPUT_ROOT / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Dataset YAML saved to: {yaml_path}")
print(f"Classes ({nc}): {class_names}")
print()
print(yaml_path.read_text())

In [ ]:
# ============================================================
# Cell 6: Verify output structure + sample labels
# ============================================================

print("Output directory structure:")
for split_name in SPLITS:
    split_dir = OUTPUT_ROOT / split_name
    if not split_dir.exists():
        continue
    n_img = len(list((split_dir / "images").glob("*")))
    n_lbl = len(list((split_dir / "labels").glob("*.txt")))
    print(f"  {split_name}/images: {n_img}  |  {split_name}/labels: {n_lbl}")

# Show a sample label
sample_label = next((OUTPUT_ROOT / "train" / "labels").glob("*.txt"), None)
if sample_label:
    print(f"\nSample label ({sample_label.name}):")
    lines = sample_label.read_text().strip().split("\n")
    for line in lines[:10]:
        parts = line.split()
        cls_id = int(parts[0])
        cls_name = class_names.get(cls_id, "?")
        print(f"  class={cls_id} ({cls_name})  xc={parts[1]}  yc={parts[2]}  w={parts[3]}  h={parts[4]}")
    if len(lines) > 10:
        print(f"  ... ({len(lines) - 10} more lines)")

In [ ]:
# ============================================================
# Cell 7: Class distribution visualization
# ============================================================
import matplotlib.pyplot as plt
from collections import Counter

# Count classes across the training set
class_counter = Counter()
train_labels_dir = OUTPUT_ROOT / "train" / "labels"

if train_labels_dir.exists():
    for label_file in train_labels_dir.glob("*.txt"):
        for line in label_file.read_text().strip().split("\n"):
            if line.strip():
                cls_id = int(line.split()[0])
                class_counter[cls_id] += 1

    names = [class_names.get(i, str(i)) for i in sorted(class_counter.keys())]
    counts = [class_counter[i] for i in sorted(class_counter.keys())]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, counts, color="steelblue", edgecolor="black")
    ax.set_xlabel("Vehicle Class")
    ax.set_ylabel("Number of Annotations")
    ax.set_title(f"VisDrone Training Set — Class Distribution ({MODE} mode)")
    ax.bar_label(bars, fmt="%d", fontsize=8)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("/kaggle/working/class_distribution.png", dpi=150)
    plt.show()
    print(f"Total annotations: {sum(counts)}")
else:
    print("No training labels found.")

In [ ]:
# ============================================================
# Cell 8: Summary
# ============================================================
print("="*60)
print("DATASET PREPARATION COMPLETE")
print("="*60)
print(f"")
print(f"Output:       {OUTPUT_ROOT}")
print(f"YAML config:  {yaml_path}")
print(f"Mode:         {MODE} ({nc} classes)")
print(f"")
print("Next steps:")
print("  1. Save this notebook output as a Kaggle dataset")
print("     → New Dataset → name it 'aeronetra-visdrone-yolo'")
print("  2. Attach that dataset to the training notebook (02_model_training)")
print("  3. The training notebook expects the dataset at:")
print(f"     /kaggle/input/aeronetra-visdrone-yolo/visdrone_yolo/")